# P2: Monochrome to Temporal-RGB Conversion (Provenance)

Converts the raw monochrome archive (Peters et al. 2014 camera; ~2 s
cadence; `<root>/YYYY/MM/DD/HH/<timestamp>.jpg`) into the temporal
three-channel composites the model consumes: frame *t* into red, *t+1* into green,
*t+2* into blue. Static features come out gray; anything that moved between
frames shows as color.

Requires the monochrome archive (`config.MONO_ARCHIVE_DIR`). The December 
2013 to January 2016 RGB conversion already exists; the training dataset 
and the December 2015 CSV were produced from it. Run this only to convert 
new imagery.

In [ ]:
# --- Repository setup -------------------------------------------------------
# Locates config.py at the repository root so this notebook runs no matter
# which directory Jupyter was launched from.  All paths and constants live in
# config.py; edit that one file rather than anything below.
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "config.py").exists():
    if _root.parent == _root:
        raise FileNotFoundError(
            "config.py not found. Launch Jupyter from inside the repository, "
            "or place this notebook back in its original folder.")
    _root = _root.parent
sys.path.insert(0, str(_root))

import config
config.describe()


In [ ]:
import os, re
import cv2
from datetime import datetime
from pathlib import Path

# This cell asks where your archives are, then saves the answers into
# config.py so every notebook (and your next session) defaults to them; the
# values are also set for this session directly, so the save is a backup
# rather than a requirement. Press Enter at either prompt to accept the
# value shown in brackets. Paste a plain folder path; surrounding quotes and
# a leading file:// prefix are stripped automatically. A web link will not
# work here: download and unzip the archive from the Zenodo data record
# first, then give its folder path on this machine.

def ask_path(label, default, must_exist):
    raw = input(f"{label}\n  [{default}]: ").strip().strip('"').strip("'")
    if raw.lower().startswith(("http://", "https://")):
        raise ValueError(
            "That is a web link. Download and unzip the archive first, then "
            "re-run this cell and give the folder path on this machine.")
    if raw.lower().startswith("file://"):
        raw = raw[7:]
    path = (Path(raw).expanduser() if raw else Path(default)).resolve()
    if must_exist and not path.is_dir():
        raise FileNotFoundError(
            f"No folder found at {path}. Check the path and re-run this cell.")
    return path

def save_to_config(var_name, path):
    """Rewrite one assignment line in config.py; leave everything else alone."""
    cfg_file = Path(config.__file__)
    text = cfg_file.read_text(encoding="utf-8")
    new_line = f'{var_name} = Path(r"{path}")'
    new_text, n = re.subn(rf"^{var_name}\s*=.*$", lambda m: new_line,
                          text, count=1, flags=re.M)
    if n == 1 and new_text != text:
        cfg_file.write_text(new_text, encoding="utf-8")
        print(f"Saved to config.py: {new_line}")
    elif n == 1:
        print(f"config.py already has {var_name} = {path}")
    else:
        print(f"Could not find the {var_name} line in config.py; the entered "
              f"path is used for this session only.")

INPUT_ROOT = ask_path("Monochrome archive folder (the raw camera frames)",
                      config.MONO_ARCHIVE_DIR, must_exist=True)
OUTPUT_ROOT = ask_path("RGB output folder (composites are written here)",
                       config.RGB_ARCHIVE_DIR, must_exist=False)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

config.MONO_ARCHIVE_DIR = INPUT_ROOT          # this session
config.RGB_ARCHIVE_DIR = OUTPUT_ROOT
save_to_config("MONO_ARCHIVE_DIR", INPUT_ROOT)    # future sessions
save_to_config("RGB_ARCHIVE_DIR", OUTPUT_ROOT)

print(f"\nReading from: {INPUT_ROOT}")
print(f"Writing to:   {OUTPUT_ROOT}")

MAX_TRIPLET_SPAN_S = 8.0   # 3 frames at the 2 s cadence span about 4 s;
                           # 8 s tolerates jitter but rejects outages

TS_RE = re.compile(r"(\d{4})-(\d{2})-(\d{2})_(\d{2})(\d{2})(\d{2})_(\d{1,6})")

def parse_ts(name):
    """Timestamp from 'YYYY-MM-DD_HHMMSS_ffffff.jpg' (returns None if malformed)."""
    m = TS_RE.search(os.path.basename(name))
    if not m:
        return None
    y, mo, d, h, mi, s, us = m.groups()
    return datetime(int(y), int(mo), int(d), int(h), int(mi), int(s), int(us.ljust(6, "0")))

## 1. Collect and time-sort the source frames

Sorting by *parsed timestamp* (not by filename string) makes the pipeline
robust to naming irregularities; `dirnames.sort()` keeps the walk
deterministic.

In [ ]:
IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg")   # the raw camera frames are .png

frames = []
seen_ext = {}
parse_failures = 0
samples = []
for dirpath, dirnames, filenames in os.walk(INPUT_ROOT):
    dirnames.sort()
    for name in filenames:
        ext = os.path.splitext(name)[1].lower()
        seen_ext[ext] = seen_ext.get(ext, 0) + 1
        if len(samples) < 5:
            samples.append(name)
        if ext in IMAGE_EXTENSIONS:
            ts = parse_ts(name)
            if ts is not None:
                frames.append((ts, os.path.join(dirpath, name)))
            else:
                parse_failures += 1
frames.sort(key=lambda x: x[0])
print(f"{len(frames):,} monochrome frames indexed")

if not frames:
    print("\nNothing matched, so here is what the walk actually found:")
    print(f"  files by extension: {seen_ext or 'no files at all'}")
    print(f"  sample filenames: {samples}")
    print(f"  image files whose names failed timestamp parsing: {parse_failures}")
    print("  Expected name pattern: YYYY-MM-DD_HHMMSS_ffffff followed by the extension.")
elif parse_failures:
    print(f"note: {parse_failures:,} image files skipped (names did not parse as timestamps)")

## 2. Convert to RGB

Output layout mirrors the input (`YYYY/MM/DD/HH/`), named by the *first*
frame's timestamp. OpenCV stores channels blue-green-red, so
`cv2.merge([blue, green, red])` places frame *t* in the red channel of the
saved image: matching Figure 3 of the paper.

In [ ]:
written = skipped_gap = 0
for i in range(len(frames) - 2):
    (t0, p0), (t1, p1), (t2, p2) = frames[i], frames[i + 1], frames[i + 2]
    if (t2 - t0).total_seconds() > MAX_TRIPLET_SPAN_S:
        skipped_gap += 1
        continue                        # frames straddle an outage: no composite
    red   = cv2.cvtColor(cv2.imread(p0), cv2.COLOR_BGR2GRAY)
    green = cv2.cvtColor(cv2.imread(p1), cv2.COLOR_BGR2GRAY)
    blue  = cv2.cvtColor(cv2.imread(p2), cv2.COLOR_BGR2GRAY)
    merged = cv2.merge([blue, green, red])

    stem = os.path.splitext(os.path.basename(p0))[0]
    out_dir = OUTPUT_ROOT / f"{t0:%Y}" / f"{t0:%m}" / f"{t0:%d}" / f"{t0:%H}"
    out_dir.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out_dir / f"{stem}.jpg"), merged)
    written += 1
    if written % 50000 == 0:
        print(f"  {written:,} composites written...")

print(f"Done: {written:,} composites written; {skipped_gap:,} triplets skipped at gaps.")
